In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY not found")

print("Environment configured successfully.")

Environment configured successfully.


In [2]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

In [3]:
embedding_model = OpenAIEmbeddings(
    api_key=os.environ["OPENAI_API_KEY"],
    model="text-embedding-3-small"
)

print("Embedding model initialized.")

Embedding model initialized.


In [4]:
vector_store = Chroma(
    collection_name="CSV_RAG_BASELINE",
    persist_directory="../chroma_db",
    embedding_function=embedding_model
)

print("Connected to ChromaDB.")

Connected to ChromaDB.


In [5]:
document_count = vector_store._collection.count()

print(f"Vectors stored in ChromaDB: {document_count}")

Vectors stored in ChromaDB: 1538


In [6]:
question = "Which product is associated with Renal Cell Carcinoma?"

print("Question:", question)

Question: Which product is associated with Renal Cell Carcinoma?


In [7]:
results_k1 = vector_store.similarity_search_with_relevance_scores(
    question,
    k=1
)

print("Number of results:", len(results_k1))

for rank, (doc, score) in enumerate(results_k1, start=1):
    print("=" * 80)
    print(f"Rank: {rank}")
    print(f"Relevance Score: {score:.4f}")
    print(f"Metadata: {doc.metadata}")
    print(doc.page_content[:500])

Number of results: 1
Rank: 1
Relevance Score: 0.3642
Metadata: {'row': 135, 'source': '..\\data\\Pharma_Sales_Long.csv'}
Notes: Product Overview: WELIREG is used in Renal Cell Carcinoma. Clinical discussion covered approved indications, patient eligibility, efficacy, safety, monitoring, treatment pathway, and physician education. Sales discussion included customer questions, objections, market access, competitor comparison, follow-up planning, CRM updates, scientific literature sharing, compliant promotion, and future engagement opportunities. Territory insights included prescription trends, customer behavior,


In [8]:
results_k3 = vector_store.similarity_search_with_relevance_scores(
    question,
    k=3
)

print("Number of results:", len(results_k3))

for rank, (doc, score) in enumerate(results_k3, start=1):
    print("=" * 80)
    print(f"Rank: {rank}")
    print(f"Relevance Score: {score:.4f}")
    print(f"Metadata: {doc.metadata}")
    print(doc.page_content[:500])

Number of results: 3
Rank: 1
Relevance Score: 0.3642
Metadata: {'source': '..\\data\\Pharma_Sales_Long.csv', 'row': 85}
Notes: Product Overview: WELIREG is used in Renal Cell Carcinoma. Clinical discussion covered approved indications, patient eligibility, efficacy, safety, monitoring, treatment pathway, and physician education. Sales discussion included customer questions, objections, market access, competitor comparison, follow-up planning, CRM updates, scientific literature sharing, compliant promotion, and future engagement opportunities. Territory insights included prescription trends, customer behavior,
Rank: 2
Relevance Score: 0.3642
Metadata: {'source': '..\\data\\Pharma_Sales_Long.csv', 'row': 135}
Notes: Product Overview: WELIREG is used in Renal Cell Carcinoma. Clinical discussion covered approved indications, patient eligibility, efficacy, safety, monitoring, treatment pathway, and physician education. Sales discussion included customer questions, objections, market access,

In [9]:
results_k5 = vector_store.similarity_search_with_relevance_scores(
    question,
    k=5
)

print("Number of results:", len(results_k5))

for rank, (doc, score) in enumerate(results_k5, start=1):
    print("=" * 80)
    print(f"Rank: {rank}")
    print(f"Relevance Score: {score:.4f}")
    print(f"Metadata: {doc.metadata}")
    print(doc.page_content[:500])

Number of results: 5
Rank: 1
Relevance Score: 0.3642
Metadata: {'row': 85, 'source': '..\\data\\Pharma_Sales_Long.csv'}
Notes: Product Overview: WELIREG is used in Renal Cell Carcinoma. Clinical discussion covered approved indications, patient eligibility, efficacy, safety, monitoring, treatment pathway, and physician education. Sales discussion included customer questions, objections, market access, competitor comparison, follow-up planning, CRM updates, scientific literature sharing, compliant promotion, and future engagement opportunities. Territory insights included prescription trends, customer behavior,
Rank: 2
Relevance Score: 0.3642
Metadata: {'row': 105, 'source': '..\\data\\Pharma_Sales_Long.csv'}
Notes: Product Overview: WELIREG is used in Renal Cell Carcinoma. Clinical discussion covered approved indications, patient eligibility, efficacy, safety, monitoring, treatment pathway, and physician education. Sales discussion included customer questions, objections, market access,

In [10]:
comparison = []

for experiment_id, k, results in [
    ("SS001", 1, results_k1),
    ("SS002", 3, results_k3),
    ("SS003", 5, results_k5)
]:
    
    scores = [score for _, score in results]

    comparison.append({
        "Experiment_ID": experiment_id,
        "K": k,
        "Results_Returned": len(results),
        "Top_Score": max(scores) if scores else None,
        "Average_Score": sum(scores) / len(scores) if scores else None
    })

In [11]:
import pandas as pd

comparison_df = pd.DataFrame(comparison)

comparison_df

,Experiment_ID,K,Results_Returned,Top_Score,Average_Score
0,SS001,1,1,0.364235,0.364235
1,SS002,3,3,0.364235,0.364235
2,SS003,5,5,0.364235,0.364235


In [12]:
for experiment_id, k, results in [
    ("SS001", 1, results_k1),
    ("SS002", 3, results_k3),
    ("SS003", 5, results_k5)
]:

    print("=" * 80)
    print(f"{experiment_id} | k = {k}")

    for rank, (doc, score) in enumerate(results, start=1):

        print(
            f"Rank {rank} | "
            f"Row {doc.metadata.get('row')} | "
            f"Score {score:.4f}"
        )

SS001 | k = 1
Rank 1 | Row 135 | Score 0.3642
SS002 | k = 3
Rank 1 | Row 85 | Score 0.3642
Rank 2 | Row 135 | Score 0.3642
Rank 3 | Row 186 | Score 0.3642
SS003 | k = 5
Rank 1 | Row 85 | Score 0.3642
Rank 2 | Row 105 | Score 0.3642
Rank 3 | Row 109 | Score 0.3642
Rank 4 | Row 135 | Score 0.3642
Rank 5 | Row 186 | Score 0.3642


In [13]:
from pathlib import Path
import pandas as pd

experiment_log_path = Path("../experiments/experiment_log.csv")

similarity_results = pd.DataFrame([
    {
        "Experiment_ID": "SS001",
        "Area": "Similarity Search",
        "Configuration": "k=1",
        "Question": question,
        "Documents": 300,
        "Chunks": 1538,
        "Top_K": 1,
        "Results": "Row 135 | Score 0.3642",
        "Observation": "One relevant WELIREG result retrieved",
        "Conclusion": "k=1 provides the minimum retrieval context"
    },
    {
        "Experiment_ID": "SS002",
        "Area": "Similarity Search",
        "Configuration": "k=3",
        "Question": question,
        "Documents": 300,
        "Chunks": 1538,
        "Top_K": 3,
        "Results": "Rows 85, 135, 186 | Score 0.3642",
        "Observation": "Three WELIREG records retrieved with identical scores",
        "Conclusion": "Additional results did not provide score diversity"
    },
    {
        "Experiment_ID": "SS003",
        "Area": "Similarity Search",
        "Configuration": "k=5",
        "Question": question,
        "Documents": 300,
        "Chunks": 1538,
        "Top_K": 5,
        "Results": "Rows 85, 105, 109, 135, 186 | Score 0.3642",
        "Observation": "Five WELIREG records retrieved with identical scores",
        "Conclusion": "Higher k increases context but may introduce redundant information"
    }
])

similarity_results


,Experiment_ID,Area,Configuration,Question,Documents,Chunks,Top_K,Results,Observation,Conclusion
0,SS001,Similarity Search,k=1,Which product is associated with Renal Cell Ca...,300,1538,1,Row 135 | Score 0.3642,One relevant WELIREG result retrieved,k=1 provides the minimum retrieval context
1,SS002,Similarity Search,k=3,Which product is associated with Renal Cell Ca...,300,1538,3,"Rows 85, 135, 186 | Score 0.3642",Three WELIREG records retrieved with identical...,Additional results did not provide score diver...
2,SS003,Similarity Search,k=5,Which product is associated with Renal Cell Ca...,300,1538,5,"Rows 85, 105, 109, 135, 186 | Score 0.3642",Five WELIREG records retrieved with identical ...,Higher k increases context but may introduce r...


In [14]:
similarity_results.to_csv(
    experiment_log_path,
    mode="a",
    header=not experiment_log_path.exists(),
    index=False
)

print(f"Similarity search experiments saved to: {experiment_log_path}")

Similarity search experiments saved to: ..\experiments\experiment_log.csv
